# Lab 04 — Sistemas de 1ª e 2ª ordem: polos, amortecimento e especificações

**Unidade II — Elementos essenciais em um sistema de controle** · conteúdo 2.1 do PPC

**Objetivos:**
1. Relacionar a posição dos polos no plano $s$ com a forma da resposta temporal;
2. Verificar experimentalmente as fórmulas de $M_p$, $t_p$, $t_s$ e $t_r$;
3. Traduzir especificações de desempenho em regiões admissíveis do plano $s$;
4. Estudar o efeito da realimentação proporcional sobre sistemas de 1ª ordem.

**Referências:** Åström & Murray (FBS), cap. 5 · Ogata, cap. 5 · Dorf & Bishop, cap. 5.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    import control as ct
    print("python-control", ct.__version__)
except ImportError:
    %pip install control
    import control as ct

## 1. Primeira ordem em malha fechada: sempre estável, nunca de graça

$G(s) = \dfrac{K}{\tau s + 1}$ com controlador proporcional $K_p$ resulta em malha fechada
$$T(s) = \frac{K_p K}{\tau s + 1 + K_p K}
       = \frac{\frac{K_p K}{1 + K_p K}}{\frac{\tau}{1 + K_p K}s + 1}$$
**Mais rápido** (constante de tempo dividida por $1 + K_pK$) e com **erro de regime**
$e_\infty = \frac{1}{1+K_pK}$.

In [ ]:
K, tau = 2.0, 3.0
G1 = ct.tf([K], [tau, 1])

plt.figure(figsize=(9, 5))
t = np.linspace(0, 12, 600)
for Kp in [0.5, 1, 3, 10]:
    T_cl = ct.feedback(Kp * G1, 1)
    resp = ct.step_response(T_cl, t)
    polo = ct.poles(T_cl)[0].real
    plt.plot(resp.time, resp.outputs, lw=2,
             label=f'Kp = {Kp} (polo em {polo:.2f}, e_inf = {1/(1+Kp*K):.3f})')
plt.axhline(1, color='gray', ls='--')
plt.xlabel('Tempo [s]')
plt.ylabel('y(t)')
plt.title('1ª ordem realimentada: velocidade × erro de regime')
plt.legend()
plt.grid(True)
plt.show()

**Conclusões:** aumentar $K_p$ (i) desloca o polo para a esquerda (mais rápido), (ii) reduz —
mas nunca zera — o erro de regime, e (iii) **jamais instabiliza** uma 1ª ordem pura.
Na prática, o limite vem da **saturação do atuador** (Unidade III).

## 2. Segunda ordem padrão: o mapa polos → resposta

$$G(s) = \frac{\omega_n^2}{s^2 + 2\zeta\omega_n s + \omega_n^2}$$

In [ ]:
wn = 2.0
zetas = [0.1, 0.3, 0.5, 0.7, 1.0, 2.0]

fig, (ax_t, ax_s) = plt.subplots(1, 2, figsize=(12, 5))
t = np.linspace(0, 8, 800)
for zeta in zetas:
    G2 = ct.tf([wn**2], [1, 2 * zeta * wn, wn**2])
    resp = ct.step_response(G2, t)
    ax_t.plot(resp.time, resp.outputs, lw=1.8, label=f'ζ = {zeta}')
    polos = ct.poles(G2)
    ax_s.plot(polos.real, polos.imag, 'x', ms=10, mew=2.5)

ax_t.axhline(1, color='gray', ls='--')
ax_t.set_xlabel('Tempo [s]'); ax_t.set_ylabel('y(t)')
ax_t.set_title('Respostas ao degrau'); ax_t.legend(); ax_t.grid(True)

ax_s.axhline(0, color='k', lw=0.5); ax_s.axvline(0, color='k', lw=0.5)
ax_s.set_xlabel('Re(s)'); ax_s.set_ylabel('Im(s)')
ax_s.set_title('Polos no plano s (mesma cor)'); ax_s.grid(True)
plt.show()

**Leitura do plano s:** polos afastados do eixo imaginário ⟹ decaimento rápido;
polos próximos do eixo real ⟹ pouca oscilação. O ângulo do polo com o eixo real negativo
é $\arccos\zeta$.

## 3. Verificando as fórmulas de especificação

$$M_p = 100\,e^{-\pi\zeta/\sqrt{1-\zeta^2}}\,\%\qquad t_p = \frac{\pi}{\omega_d}\qquad t_s \approx \frac{4}{\zeta\omega_n}$$

In [ ]:
print(f"{'zeta':>5s} | {'Mp teor.[%]':>11s} | {'Mp medido[%]':>12s} | "
      f"{'ts teor.[s]':>11s} | {'ts medido[s]':>12s}")
print('-' * 65)
for zeta in [0.2, 0.4, 0.6, 0.8]:
    G2 = ct.tf([wn**2], [1, 2 * zeta * wn, wn**2])
    info = ct.step_info(G2)
    Mp_teo = 100 * np.exp(-np.pi * zeta / np.sqrt(1 - zeta**2))
    ts_teo = 4 / (zeta * wn)
    print(f"{zeta:5.1f} | {Mp_teo:11.2f} | {info['Overshoot']:12.2f} | "
          f"{ts_teo:11.2f} | {info['SettlingTime']:12.2f}")

As fórmulas conferem com boa precisão ($t_s$ é aproximação — a `step_info` usa o critério
de faixa de 2 % exato, a fórmula usa a envoltória exponencial).

## 4. Do requisito ao projeto: região admissível no plano s

**Especificações de exemplo:** $M_p \le 10\,\%$ e $t_s \le 2$ s.

- $M_p \le 10\,\% \Rightarrow \zeta \ge 0{,}59$ ⟹ polos dentro do cone $|\arg| \le \arccos(0{,}59) \approx 54°$;
- $t_s \le 2 \Rightarrow \zeta\omega_n \ge 2$ ⟹ polos à esquerda da reta $\mathrm{Re}(s) = -2$.

In [ ]:
zeta_min = 0.59
sigma_min = 2.0

fig, ax = plt.subplots(figsize=(7, 6))
# reta de tempo de acomodação
ax.axvline(-sigma_min, color='C1', lw=2, label=r'$t_s$: Re(s) = -2')
# cone de amortecimento
theta = np.arccos(zeta_min)
r = np.linspace(0, 8, 50)
ax.plot(-r * np.cos(theta), r * np.sin(theta), 'C2', lw=2, label=r'$M_p$: ζ = 0,59')
ax.plot(-r * np.cos(theta), -r * np.sin(theta), 'C2', lw=2)

# região admissível (hachura simples por pontos)
xx, yy = np.meshgrid(np.linspace(-8, 0.5, 200), np.linspace(-6, 6, 200))
adm = (xx <= -sigma_min) & (np.abs(np.arctan2(np.abs(yy), -xx)) <= theta)
ax.contourf(xx, yy, adm, levels=[0.5, 1.5], colors=['C0'], alpha=0.2)

# um par de polos candidato dentro da região
polos_cand = np.array([-3 + 3j, -3 - 3j])
ax.plot(polos_cand.real, polos_cand.imag, 'kx', ms=12, mew=3, label='polos escolhidos')

ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
ax.set_xlabel('Re(s)'); ax.set_ylabel('Im(s)')
ax.set_title('Região admissível para Mp ≤ 10% e ts ≤ 2 s')
ax.legend(loc='lower left'); ax.grid(True)
plt.show()

In [ ]:
# conferindo o par de polos escolhido: s = -3 +/- 3j  =>  wn e zeta correspondentes
wn_c = np.abs(polos_cand[0])
zeta_c = -polos_cand[0].real / wn_c
G_cand = ct.tf([wn_c**2], [1, 2 * zeta_c * wn_c, wn_c**2])
info = ct.step_info(G_cand)
print(f"wn = {wn_c:.2f}, zeta = {zeta_c:.2f} -> "
      f"Mp = {info['Overshoot']:.1f} %, ts = {info['SettlingTime']:.2f} s")

As especificações são atendidas. **Este raciocínio inverso — especificação → polos → controlador —
é o coração do projeto de controle** e será usado com PID na Unidade IV.

## 5. Efeito de um polo adicional e de um zero

Sistemas reais raramente são 2ª ordem pura. Vale conhecer duas perturbações comuns:

In [ ]:
G_base = ct.tf([wn**2], [1, 2 * 0.5 * wn, wn**2])       # 2ª ordem, zeta = 0.5
G_polo = G_base * ct.tf([1], [0.5, 1])                   # polo extra em -2
G_zero_rhp = G_base * ct.tf([-0.5, 1], [1])              # zero de fase não mínima em +2

t = np.linspace(0, 8, 800)
plt.figure(figsize=(9, 4.5))
for sys, nome in [(G_base, '2ª ordem pura'),
                  (G_polo, '+ polo extra (mais lento)'),
                  (G_zero_rhp, '+ zero no SPD (resposta inversa)')]:
    resp = ct.step_response(sys, t)
    plt.plot(resp.time, resp.outputs, lw=2, label=nome)
plt.axhline(1, color='gray', ls='--')
plt.xlabel('Tempo [s]'); plt.ylabel('y(t)')
plt.title('Dinâmicas adicionais alteram a resposta padrão')
plt.legend(); plt.grid(True)
plt.show()

O **zero no semiplano direito** produz *resposta inversa* (a saída inicialmente vai na direção
errada) — comportamento típico de alguns processos térmicos e hidráulicos, e um limitador
fundamental de desempenho.

---
> **🖼️ Figuras de apoio nos livros:**
> - Ogata, **Figura 5.8** — especificações de resposta transitória ($t_d, t_r, t_p, M_p, t_s$) sobre a curva ao degrau. Cap. 5, seção 'Especificações de resposta transitória', **p. 154** (p. 165 do PDF).
> - Ogata, **Figura 5.22** — família de respostas ao degrau unitário para $\zeta = 0; 0{,}2; \dots; 1$ ($\omega_n = 1$). Cap. 5, **p. 174** (p. 185 do PDF).
> - Nise, **Figura 4.14** — especificações da resposta subamortecida de 2ª ordem ($T_r, T_p, \%UP, T_s$). Cap. 4, p. 264 do arquivo PDF (a cópia digital não exibe o nº impresso).
> - Penedo, **Figura 4.4** — localização dos polos complexos conjugados de 2ª ordem no plano complexo. Cap. 4, p. 42 do arquivo PDF.

## Exercícios (relatório do Lab 04)

**E1.** Para o motor CC completo do Lab 01, calcule $\zeta$ e $\omega_n$ com `ct.damp` e
preveja $M_p$ e $t_s$ pelas fórmulas. Confira com `ct.step_info`.

**E2.** Projete (escolha $\zeta$, $\omega_n$) um sistema de 2ª ordem com $M_p \le 5\,\%$ e
$t_p \le 0{,}5$ s. Mostre a região admissível e valide por simulação.

**E3.** Adicione ao sistema do E2 um polo extra em $s = -p$ e encontre, por tentativa, o menor
$p$ para o qual as especificações continuam atendidas. Enuncie uma regra prática sobre
"polos dominantes".

**E4.** Mostre por simulação que, para $\zeta \ge 1$, não há sobressinal, e que o sistema
criticamente amortecido é o mais rápido **sem** sobressinal.

In [ ]:
# E1 — sua solução aqui

In [ ]:
# E2 — sua solução aqui

In [ ]:
# E3 — sua solução aqui

In [ ]:
# E4 — sua solução aqui